# LLM příznaky z názvu a popisu kampaně (AI část, cesta A)

Tento notebook využívá jazykový model na školním endpointu k tomu, aby na základě názvu (`name`) a krátkého popisu (`blurb`) každé kampaně vyplnil několik jednoduchých příznaků. [Hlavní notebook](../main.ipynb) je následně načítá z CSV a porovnává modely s těmito příznaky a bez nich.

Postup má dva kroky:
1. Discovery: model obdrží vzorek textů a sám navrhne příznaky, které by mohly souviset s úspěchem kampaně. Vedle toho používáme vlastní, ručně navržený dotazník.
2. Generování: model podle zafixovaného dotazníku vyplní příznaky pro každou kampaň.

Zásady, které notebook dodržuje:
- Model vidí pouze `name` a `blurb`. Stav kampaně, vybrané částky ani počet podporovatelů do promptu nevstupují. Sloupec `state` slouží pouze k výběru dokončených kampaní a ihned poté jej odstraňujeme.
- Všech 478 kampaní z discovery a pilotu je i s jejich tvůrci vždy v trénovací sadě (hlavní notebook, sekce 3.5), testovací sada tedy na návrhu dotazníků nezávisí. Výběr pouze z ledna až října je pozůstatkem původního plánu s testovací sadou z listopadu a prosince.
- API klíč se načítá ze souboru `.env` nebo z proměnné prostředí `LLM_API_KEY`.
- Každá odpověď modelu se ukládá do cache ve složce `_data/ai/cache/`, opakované spuštění tedy již zpracované dávky znovu neodesílá.

Notebook má dva režimy. `RUN_MODE = "pilot"` zpracuje vzorek přibližně 300 kampaní pro ověření postupu, `RUN_MODE = "full"` zpracuje všechny dokončené kampaně a vytvoří CSV pro hlavní notebook. Uložené výstupy v tomto souboru pocházejí z plného běhu (14. 11. 2026).

Potřebné balíčky: `pandas` a `openai` (`pip install openai pandas`). Balíček `openai` slouží jako klient pro školní LiteLLM endpoint, který používá rozhraní kompatibilní s OpenAI.

<a id="s0"></a>
## 0. Nastavení

Všechny nastavitelné hodnoty jsou soustředěny v této buňce. Pro pilot jsou vhodné výchozí hodnoty, pro plný běh jsme nastavili `RUN_MODE` na `"full"` a `MAX_WORKERS` na 4.

In [ ]:
RUN_MODE = "full"             # "pilot" = vzorek pro ladění, "full" = všechny dokončené projekty
PILOT_SIZE = 300              # velikost pilotního vzorku
DISCOVERY_SIZE = 150          # kolik textů uvidí model při navrhování příznaků
TUNING_MONTHS = range(1, 11)  # leden–říjen 2025; z těchto měsíců bereme vzorky pro discovery a pilot
SEED = 42                     # pevný seed

SCHEMAS_TO_RUN = ["manual", "discovered"]  # která schémata generovat
REDISCOVER = False            # True = znovu nechat model navrhnout příznaky (přepíše schema_discovered.json)
PROMPT_VERSION = "v1"         # při změně promptu zvýšit, aby se nepoužila stará cache

BATCH_SIZE = 20               # počet projektů v jednom dotazu
MAX_WORKERS = 4               # počet souběžných dotazů; při chybě 429 (rate limit) snižte na 1
MAX_ATTEMPTS = 3              # počet pokusů o jednu dávku
CONSISTENCY_SIZE = 60         # kolik pilotních projektů se pro kontrolu konzistence pošle podruhé

In [ ]:
import hashlib
import json
import math
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display

# Funguje při spuštění z kořene projektu i ze složky notebooks.
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "_data").is_dir() else Path.cwd().parent
PREPARED_DIR = PROJECT_ROOT / "_data" / "prepared"
AI_DIR = PROJECT_ROOT / "_data" / "ai"
CACHE_DIR = AI_DIR / "cache"
AI_DIR.mkdir(parents=True, exist_ok=True)


def load_env_file(path):
    """Načte řádky KLÍČ=hodnota ze souboru .env do proměnných prostředí."""
    if not path.is_file():
        return
    for line in path.read_text(encoding="utf-8-sig").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env_file(PROJECT_ROOT / ".env")
BASE_URL = os.environ.get("LLM_BASE_URL", "https://litellm.vse.cz/v1/")
CHAT_MODEL = os.environ.get("LLM_CHAT_MODEL", "qwen3.6-35b")

if not os.environ.get("LLM_API_KEY"):
    raise RuntimeError(
        "Chybí klíč LLM_API_KEY. Vytvořte v kořeni projektu soubor .env s řádkem "
        "LLM_API_KEY=<llm api klíč>."
    )
assert RUN_MODE in ("pilot", "full"), "RUN_MODE musí být 'pilot' nebo 'full'."
print(f"Projekt: {PROJECT_ROOT.resolve().name}")
print(f"Endpoint: {BASE_URL} · model: {CHAT_MODEL} · režim: {RUN_MODE}")

Projekt: kickstarter-llm-run
Endpoint: https://litellm.vse.cz/v1/ · model: qwen3.6-35b · režim: full


<a id="s1"></a>
## 1. Spojení s modelem

Buňka ověří klíč a spojení a zjistí, které volby endpoint podporuje. Nejprve zkusí odpověď v režimu JSON s vypnutým „přemýšlením“ modelu Qwen, které zpracování zrychluje a zlevňuje. Pokud endpoint některou volbu odmítne, buňka ji vypne a pokus zopakuje.

In [ ]:
import openai
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=os.environ["LLM_API_KEY"], timeout=180, max_retries=2)
API_OPTIONS = {"json_mode": True, "disable_thinking": True}


def call_model(messages, use_json=True):
    # Jeden dotaz na chat model. Vrací text odpovědi a údaje o čase a tokenech
    kwargs = {"model": CHAT_MODEL, "messages": messages, "temperature": 0}
    if use_json and API_OPTIONS["json_mode"]:
        kwargs["response_format"] = {"type": "json_object"}
    if API_OPTIONS["disable_thinking"]:
        kwargs["extra_body"] = {"chat_template_kwargs": {"enable_thinking": False}}
    started = time.perf_counter()
    response = client.chat.completions.create(**kwargs)
    seconds = time.perf_counter() - started
    content = response.choices[0].message.content or ""
    content = re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL).strip()
    usage = getattr(response, "usage", None)
    return content, {
        "seconds": round(seconds, 3),
        "prompt_tokens": getattr(usage, "prompt_tokens", None),
        "completion_tokens": getattr(usage, "completion_tokens", None),
    }


def parse_json_object(text):
    # Najde v odpovědi JSON objekt
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    for candidate in (text, text[text.find("{"): text.rfind("}") + 1]):
        try:
            value = json.loads(candidate)
        except ValueError:
            continue
        if isinstance(value, dict):
            return value
    return None


def probe_endpoint():
    messages = [{"role": "user", "content": 'Return the JSON object {"status": "ok"} and nothing else.'}]
    last_error = None
    for json_mode, disable_thinking in ((True, True), (True, False), (False, True), (False, False)):
        API_OPTIONS.update(json_mode=json_mode, disable_thinking=disable_thinking)
        try:
            content, meta = call_model(messages)
        except openai.AuthenticationError as error:
            raise RuntimeError("Endpoint odmítl klíč (401). Zkontrolujte LLM_API_KEY v souboru .env.") from error
        except openai.APIConnectionError as error:
            raise RuntimeError(
                f"Nepodařilo se spojit s {BASE_URL}. Zkuste školní síť nebo VPN, případně jupyter.vse.cz."
            ) from error
        except openai.BadRequestError as error:
            last_error = error
            continue
        if parse_json_object(content) is not None:
            return content, meta
        last_error = ValueError(f"Model nevrátil JSON: {content[:200]!r}")
    raise RuntimeError("Endpoint nevrátil použitelnou odpověď v žádném režimu.") from last_error


probe_content, probe_meta = probe_endpoint()
print(f"Odpověď: {probe_content} · {probe_meta['seconds']} s")
print(f"Režim JSON: {API_OPTIONS['json_mode']} · vypnuté přemýšlení: {API_OPTIONS['disable_thinking']}")

Odpověď: {"status": "ok"} · 0.265 s
Režim JSON: True · vypnuté přemýšlení: True


<a id="s2"></a>
## 2. Texty kampaní

Načítáme pouze sloupce potřebné k výběru kampaní. Stejně jako v hlavním notebooku pracujeme s dokončenými kampaněmi (`successful`, `failed`) bez konfliktu mezi snapshoty. Hlavní notebook navíc vyřazuje 8 kampaní delších než 60 dní, zde je tedy kampaní o 8 více a připojení podle `id` funguje bez omezení. Sloupec `state` slouží pouze k výběru a ihned poté jej odstraňujeme, výsledek kampaně se tedy k modelu nedostane.

In [4]:
part_paths = sorted(PREPARED_DIR.glob("kickstarter_projects_2025_part_[0-9][0-9][0-9].csv"))
if not part_paths:
    raise FileNotFoundError(f"Ve složce {PREPARED_DIR} chybí připravené části CSV.")

columns = ["id", "name", "blurb", "category", "launched_at", "state", "snapshot_has_state_or_launch_conflict"]
raw = pd.concat(
    [pd.read_csv(path, dtype=str, keep_default_na=False, usecols=columns) for path in part_paths],
    ignore_index=True,
)
keep = raw["state"].isin(["successful", "failed"]) & raw["snapshot_has_state_or_launch_conflict"].str.lower().ne("true")


def main_category(raw_value):
    try:
        category = json.loads(raw_value)
    except ValueError:
        return "__MISSING__"
    if not isinstance(category, dict):
        return "__MISSING__"
    return category.get("parent_name") or category.get("name") or "__MISSING__"


texts = raw.loc[keep, ["id", "name", "blurb", "category", "launched_at"]].copy()
texts["id"] = texts["id"].str.strip()
texts["name"] = texts["name"].str.strip()
texts["blurb"] = texts["blurb"].str.strip()
texts["main_category"] = texts["category"].map(main_category)
texts["launch_month"] = pd.to_datetime(pd.to_numeric(texts["launched_at"]), unit="s", utc=True).dt.month
texts = texts.drop(columns=["category", "launched_at"]).sort_values("id").reset_index(drop=True)
del raw, keep  # výsledek kampaně dál v paměti není

assert "state" not in texts.columns
assert texts["id"].is_unique, "Duplicitní ID projektu."
print(f"Dokončených projektů: {len(texts):,}")
print(f"Z toho spuštěných v lednu–říjnu (pro ladění): {texts['launch_month'].isin(TUNING_MONTHS).sum():,}")

Dokončených projektů: 26,635
Z toho spuštěných v lednu–říjnu (pro ladění): 22,976


<a id="s2-1"></a>
### 2.1 Vzorky pro discovery a pilot

Oba vzorky vybíráme z ledna až října, stratifikovaně podle hlavní kategorie. Každá kategorie získá poměrnou část, nejméně však 5 kampaní, aby byly zastoupeny i malé kategorie jako Dance nebo Journalism. Z tohoto důvodu má discovery 169 kampaní místo 150 a pilot 309 místo 300. Vzorky se nepřekrývají, dotazníky tedy ověřujeme na jiných textech, než ze kterých je model navrhl. Seznamy ID ukládáme kvůli opakovatelnosti a hlavní notebook podle nich tyto kampaně zařazuje do trénovací sady.

In [5]:
def stratified_sample(frame, size, seed, min_per_group=5, exclude=()):
    pool = frame.loc[~frame["id"].isin(set(exclude))]
    shares = pool["main_category"].value_counts(normalize=True)
    parts = []
    for category, share in shares.items():
        group = pool.loc[pool["main_category"].eq(category)]
        count = min(len(group), max(min_per_group, round(size * share)))
        parts.append(group.sample(count, random_state=seed))
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)


tuning_pool = texts.loc[texts["launch_month"].isin(TUNING_MONTHS)]
discovery_sample = stratified_sample(tuning_pool, DISCOVERY_SIZE, SEED)
pilot_sample = stratified_sample(tuning_pool, PILOT_SIZE, SEED + 1, exclude=discovery_sample["id"])
assert not set(discovery_sample["id"]) & set(pilot_sample["id"])

discovery_sample[["id"]].to_csv(AI_DIR / "discovery_sample_ids.csv", index=False)
pilot_sample[["id"]].to_csv(AI_DIR / "pilot_sample_ids.csv", index=False)

coverage = pd.DataFrame({
    "Discovery": discovery_sample["main_category"].value_counts(),
    "Pilot": pilot_sample["main_category"].value_counts(),
    "Leden–říjen celkem": tuning_pool["main_category"].value_counts(),
}).fillna(0).astype(int).sort_values("Leden–říjen celkem", ascending=False)
print(f"Discovery: {len(discovery_sample)} projektů · pilot: {len(pilot_sample)} projektů")
display(coverage)

Discovery: 169 projektů · pilot: 309 projektů


,Discovery,Pilot,Leden–říjen celkem
main_category,,,
Games,35,69,5286
Publishing,21,42,3238
Comics,17,35,2641
Film & Video,15,29,2237
Technology,14,27,2100
Design,13,26,2027
Art,10,20,1548
Music,7,13,1016
Fashion,6,12,894


<a id="s3"></a>
## 3. Dotazníky (schémata příznaků)

Schéma je seznam příznaků, u každého je uveden název, typ a popis pro model:
- `category`: pevný seznam povolených hodnot,
- `boolean`: ano/ne,
- `integer`: číslo na stupnici, například 1–5.

Odpověď mimo povolené hodnoty ukládáme jako chybějící a nijak ji nedoplňujeme.

<a id="s3-1"></a>
### 3.1 Ruční dotazník

Dotazník jsme navrhli na základě průzkumu dat a úvahy, které vlastnosti lze z názvu a popisu rozpoznat na první pohled. Například u kategorie Food uspělo pouze 27 % kampaní a mezi neúspěšnými převažují žádosti o záchranu nebo otevření podniku.

In [6]:
MANUAL_SCHEMA = {
    "name": "manual",
    "features": [
        {"name": "funding_purpose", "type": "category",
         "values": ["product", "creative_work", "business_operations", "charity_or_cause", "event", "other"],
         "description": "What the money is mainly for: manufacturing a product for backers; creating a creative work "
                        "(book, comic, game, film, album, art); opening, running or saving a business or venue; "
                        "a charitable or community cause; organising an event; or other."},
        {"name": "has_concrete_deliverable", "type": "boolean",
         "description": "True if the text makes clear that backers will receive a specific thing "
                        "(a copy, a product, a reward); false if it only asks for support."},
        {"name": "project_stage", "type": "category", "values": ["idea", "in_progress", "finished", "unclear"],
         "description": "How far along the project appears to be according to the text."},
        {"name": "appeal_type", "type": "category", "values": ["offer_to_backers", "personal_plea", "mixed"],
         "description": "Whether the text sells something to backers (offer) or asks for help for the creator (plea)."},
        {"name": "specificity", "type": "integer", "min": 1, "max": 5,
         "description": "How concrete the description is: 1 = vague slogan, 5 = precise what, how and for whom."},
        {"name": "language", "type": "category", "values": ["en", "de", "fr", "es", "it", "nl", "ja", "zh", "other"],
         "description": "Main language of the title and description."},
    ],
}

In [7]:
ALLOWED_TYPES = {"category", "boolean", "integer"}


def check_schema(schema):
    """Vrátí očištěné schéma a seznam problémů; neplatné příznaky vynechá."""
    clean, problems, seen = [], [], set()
    for feature in schema.get("features", []):
        name = str(feature.get("name", "")).strip().lower()
        kind = feature.get("type")
        if not re.fullmatch(r"[a-z][a-z0-9_]{1,40}", name) or name in seen:
            problems.append(f"{name!r}: neplatný nebo opakovaný název")
            continue
        if kind not in ALLOWED_TYPES:
            problems.append(f"{name}: nepodporovaný typ {kind!r}")
            continue
        item = {"name": name, "type": kind, "description": str(feature.get("description", "")).strip()}
        if kind == "category":
            values = list(dict.fromkeys(str(v).strip().lower() for v in feature.get("values", []) if str(v).strip()))
            if len(values) < 2:
                problems.append(f"{name}: kategorie potřebuje aspoň 2 hodnoty")
                continue
            item["values"] = values
        if kind == "integer":
            try:
                low, high = int(feature.get("min", 1)), int(feature.get("max", 5))
            except (TypeError, ValueError):
                problems.append(f"{name}: neplatné min/max")
                continue
            if low >= high:
                problems.append(f"{name}: min musí být menší než max")
                continue
            item["min"], item["max"] = low, high
        seen.add(name)
        clean.append(item)
    return {"name": schema.get("name"), "features": clean}, problems


def show_schema(schema):
    rows = []
    for feature in schema["features"]:
        if feature["type"] == "category":
            allowed = " | ".join(feature["values"])
        elif feature["type"] == "boolean":
            allowed = "true | false"
        else:
            allowed = f"{feature['min']}–{feature['max']}"
        rows.append((feature["name"], feature["type"], allowed, feature["description"]))
    with pd.option_context("display.max_colwidth", 200):
        display(pd.DataFrame(rows, columns=["Příznak", "Typ", "Hodnoty", "Popis"]))


MANUAL_SCHEMA, manual_problems = check_schema(MANUAL_SCHEMA)
assert not manual_problems, manual_problems
show_schema(MANUAL_SCHEMA)

,Příznak,Typ,Hodnoty,Popis
0,funding_purpose,category,product | creative_work | business_operations | charity_or_cause | event | other,"What the money is mainly for: manufacturing a product for backers; creating a creative work (book, comic, game, film, album, art); opening, running or saving a business or venue; a charitable or c..."
1,has_concrete_deliverable,boolean,true | false,"True if the text makes clear that backers will receive a specific thing (a copy, a product, a reward); false if it only asks for support."
2,project_stage,category,idea | in_progress | finished | unclear,How far along the project appears to be according to the text.
3,appeal_type,category,offer_to_backers | personal_plea | mixed,Whether the text sells something to backers (offer) or asks for help for the creator (plea).
4,specificity,integer,1–5,"How concrete the description is: 1 = vague slogan, 5 = precise what, how and for whom."
5,language,category,en | de | fr | es | it | nl | ja | zh | other,Main language of the title and description.


<a id="s3-2"></a>
### 3.2 Dotazník navržený modelem (discovery)

Model obdrží texty z discovery bez informace o výsledku a navrhne 6–10 příznaků. V promptu má zakázáno opakovat kategorii projektu a využívat informace, které v textu nejsou.

Návrh se ukládá do souboru [`_data/ai/schema_discovered.json`](../_data/ai/schema_discovered.json). Pokud soubor existuje, notebook jej pouze načte, ruční úpravy tedy zůstávají zachovány. Nový návrh vznikne pouze s nastavením `REDISCOVER = True`.

Návrh jsme jednou mírně upravili: vyřadili jsme příznak `urgency_or_scarcity` a zpřesnili definici `creator_credibility_signal`. Pravidla a důvody úprav popisuje [hlavní notebook, sekce 5.1](../main.ipynb#s5-1). Původní návrh je v [`schema_discovered_v1_raw.json`](../_data/ai/schema_discovered_v1_raw.json), změny jsou zaznamenány v `schema_discovered.json` pod klíčem `_revision`.

In [8]:
SCHEMA_PATH = AI_DIR / "schema_discovered.json"

SYSTEM_DISCOVER = (
    "You help design input features for a model that predicts, at launch time, whether a Kickstarter campaign "
    "will reach its funding goal. You only see campaign titles and short descriptions (blurbs). "
    "You do not know which campaigns succeeded."
)


def build_discovery_messages(sample):
    lines = "\n".join(f"- {row['name']} — {row['blurb']}" for row in sample[["name", "blurb"]].to_dict("records"))
    user = (
        f"Below are {len(sample)} campaigns. Propose 6 to 10 features that a careful reader could judge from the "
        "title and blurb alone and that plausibly relate to a campaign's chance of being funded.\n"
        "Rules:\n"
        "- Do not restate the project category or genre (games, comics, film, food, ...); the model already knows it.\n"
        "- Do not use anything that is not in the text (amount raised, number of backers, popularity, creator reputation).\n"
        '- Each feature is one of: "category" with 2-6 allowed values, "boolean", or "integer" on a 1-5 scale.\n'
        "- Use English snake_case names and short English descriptions that tell an annotator exactly how to decide.\n"
        'Return one JSON object: {"features": ['
        '{"name": "...", "type": "category", "values": ["...", "..."], "description": "..."}, '
        '{"name": "...", "type": "integer", "min": 1, "max": 5, "description": "..."}, '
        '{"name": "...", "type": "boolean", "description": "..."}]}\n\n'
        f"Campaigns:\n{lines}"
    )
    return [{"role": "system", "content": SYSTEM_DISCOVER}, {"role": "user", "content": user}]


if REDISCOVER or not SCHEMA_PATH.is_file():
    content, discovery_meta = call_model(build_discovery_messages(discovery_sample))
    (AI_DIR / "discovery_raw_response.txt").write_text(content, encoding="utf-8")
    proposal = parse_json_object(content)
    if proposal is None:
        raise RuntimeError("Model nevrátil platný JSON. Spusťte buňku znovu nebo viz discovery_raw_response.txt.")
    discovered, discovery_problems = check_schema({"name": "discovered", "features": proposal.get("features", [])})
    discovered["_meta"] = {
        "model": CHAT_MODEL, "prompt_version": PROMPT_VERSION, "discovery_size": len(discovery_sample),
        "seed": SEED, "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "skipped": discovery_problems, **discovery_meta,
    }
    SCHEMA_PATH.write_text(json.dumps(discovered, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Nový návrh uložen do {SCHEMA_PATH.name} ({discovery_meta['seconds']} s).")
    if discovery_problems:
        print("Vynechané návrhy:", *discovery_problems, sep="\n  ")
else:
    print(f"Načítám existující {SCHEMA_PATH.name} (nový návrh: REDISCOVER = True).")

stored_schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
DISCOVERED_SCHEMA, stored_problems = check_schema({"name": "discovered", "features": stored_schema.get("features", [])})
if stored_problems:
    print("Problémy v souboru schématu (příznaky vynechány):", *stored_problems, sep="\n  ")
if len(DISCOVERED_SCHEMA["features"]) < 3:
    print("Pozor: navržené schéma má méně než 3 platné příznaky.")
show_schema(DISCOVERED_SCHEMA)

Načítám existující schema_discovered.json (nový návrh: REDISCOVER = True).


,Příznak,Typ,Hodnoty,Popis
0,has_clear_value_proposition,boolean,true | false,"True if the blurb explicitly states a specific problem the product solves or a unique benefit it provides (e.g., 'aids hydration', 'custom insoles', 'foldable gaming table'). False if the text is ..."
1,has_tangible_reward,boolean,true | false,"True if the campaign offers a physical item or a specific digital asset (e.g., 'STL files', 'vinyl', 'book', 'pin', 'app') as a reward. False if the campaign is purely for funding an event, servic..."
2,is_series_continuation,boolean,true | false,"True if the title or blurb explicitly indicates this is a sequel, volume 2, issue #2, or part of an ongoing series (e.g., 'Volume #1', 'Part 3', 'Season 16', '2.0', 'Returns'). False if it is a de..."
3,creator_credibility_signal,category,none | awards_or_bestseller | established_brand_or_team,"Trust signals that are explicitly written in the title or blurb. 'awards_or_bestseller': the text names a specific award, prize, nomination or bestseller status (e.g., 'Eisner Award-winning', 'NY ..."
4,social_or_political_appeal,boolean,true | false,"True if the campaign explicitly leverages social identity, activism, or political causes in its core pitch (e.g., 'LGBT+', 'Black History', 'DEI initiative', 'sustainable', 'privacy-preserving', '..."
5,technical_complexity,integer,1–5,"Rate the perceived technical sophistication or barrier to entry based on keywords. 1: Simple/Analog (e.g., 'handmade', 'book', 'zine'). 2: Basic Digital/Physical (e.g., 'app', 'standard game'). 3:..."
6,has_emotional_hook,boolean,true | false,"True if the blurb uses strong emotional language, narrative hooks, or evocative imagery to engage the reader (e.g., 'heartbreak', 'grief', 'magic', 'thrills & chills', 'love', 'resilience'). False..."


<a id="s4"></a>
## 4. Generování příznaků

Kampaně odesíláme po dávkách (`BATCH_SIZE` = 20). V promptu má každá kampaň pouze pořadové číslo, název a popis, odpověď následně párujeme podle čísla. Pokud se odpověď nepodaří zpracovat nebo v ní některé kampaně chybí, dávku opakujeme (nejvýše `MAX_ATTEMPTS`krát). Do cache ukládáme pouze úspěšné dávky, neúspěšné se tedy při dalším spuštění zpracují znovu.

`temperature = 0` omezuje náhodnost odpovědí. Míru zbývající náhodnosti ukazuje kontrola konzistence v [5.3](#s5-3) (pouze v pilotu).

In [9]:
SYSTEM_GENERATE = (
    "You annotate Kickstarter crowdfunding campaigns. For each campaign you only see its title and short "
    "description (blurb), written at launch. Judge every feature from this text alone. Do not try to guess whether "
    "the campaign succeeded. If the text does not tell, choose the most neutral allowed value. "
    "Answer with one JSON object and nothing else."
)


def describe_feature(feature):
    if feature["type"] == "category":
        allowed = " | ".join(feature["values"])
    elif feature["type"] == "boolean":
        allowed = "true | false"
    else:
        allowed = f"integer {feature['min']}-{feature['max']}"
    return f'- "{feature["name"]}" ({allowed}): {feature["description"]}'


def build_generation_messages(schema, batch):
    features = "\n".join(describe_feature(feature) for feature in schema["features"])
    example = ", ".join(f'"{feature["name"]}": ...' for feature in schema["features"])
    campaigns = [{"n": number, "title": row["name"], "blurb": row["blurb"]}
                 for number, row in enumerate(batch[["name", "blurb"]].to_dict("records"), start=1)]
    user = (
        f"Features to fill in:\n{features}\n\n"
        "Return exactly this structure with one item per campaign, keeping its number n:\n"
        f'{{"items": [{{"n": 1, {example}}}, ...]}}\n\n'
        f"Campaigns:\n{json.dumps(campaigns, ensure_ascii=False, indent=0)}"
    )
    return [{"role": "system", "content": SYSTEM_GENERATE}, {"role": "user", "content": user}]


def coerce_value(feature, value):
    """Převede odpověď na povolenou hodnotu; cokoli jiného je chybějící hodnota."""
    if value is None:
        return None
    if feature["type"] == "category":
        text = str(value).strip().lower()
        return text if text in feature["values"] else None
    if feature["type"] == "boolean":
        if isinstance(value, bool):
            return value
        return {"true": True, "yes": True, "1": True, "false": False, "no": False, "0": False}.get(str(value).strip().lower())
    if isinstance(value, bool):
        return None
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    if not number.is_integer() or not feature["min"] <= number <= feature["max"]:
        return None
    return int(number)


def batch_path(schema, ids, namespace):
    payload = json.dumps({"model": CHAT_MODEL, "prompt": PROMPT_VERSION, "schema": schema["features"],
                          "ids": list(ids)}, sort_keys=True)
    key = hashlib.sha1(payload.encode("utf-8")).hexdigest()[:16]
    return CACHE_DIR / schema["name"] / namespace / f"{key}.json"


RETRYABLE_ERRORS = (openai.RateLimitError, openai.APITimeoutError, openai.APIConnectionError, openai.InternalServerError)


def run_batch(schema, batch, namespace):
    ids = batch["id"].tolist()
    path = batch_path(schema, ids, namespace)
    if path.is_file():
        record = json.loads(path.read_text(encoding="utf-8"))
        record["from_cache"] = True
        return record
    messages = build_generation_messages(schema, batch)
    problem, seconds, prompt_tokens, completion_tokens = None, 0.0, 0, 0
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            content, meta = call_model(messages)
        except RETRYABLE_ERRORS as error:
            problem = f"{type(error).__name__}: {error}"
            time.sleep(10 * attempt)
            continue
        seconds += meta["seconds"]
        prompt_tokens += meta["prompt_tokens"] or 0
        completion_tokens += meta["completion_tokens"] or 0
        parsed = parse_json_object(content)
        items = parsed.get("items") if parsed else None
        if not isinstance(items, list):
            problem = "odpověď není platný JSON se seznamem items"
            continue
        answers = {}
        for item in items:
            try:
                number = int(item.get("n"))
            except (AttributeError, TypeError, ValueError):
                continue
            if 1 <= number <= len(ids) and number not in answers:
                answers[number] = {f["name"]: coerce_value(f, item.get(f["name"])) for f in schema["features"]}
        missing = len(ids) - len(answers)
        if missing and attempt < MAX_ATTEMPTS:
            problem = f"v odpovědi chybí {missing} projektů"
            continue
        record = {
            "ids": ids,
            "rows": [{"id": pid, **answers.get(number, {})} for number, pid in enumerate(ids, start=1)],
            "requests": attempt, "seconds": round(seconds, 3),
            "prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens,
            "missing_items": missing, "raw_response": content,
        }
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(record, ensure_ascii=False), encoding="utf-8")
        record["from_cache"] = False
        return record
    return {"ids": ids, "rows": [{"id": pid} for pid in ids], "requests": MAX_ATTEMPTS, "seconds": round(seconds, 3),
            "prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens,
            "missing_items": len(ids), "error": problem, "from_cache": False}


def run_schema(schema, frame, namespace):
    """Zpracuje všechny dávky; vrací tabulku příznaků (index = id) a souhrn nákladů."""
    batches = [frame.iloc[start:start + BATCH_SIZE] for start in range(0, len(frame), BATCH_SIZE)]
    records, started = [], time.perf_counter()
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = [pool.submit(run_batch, schema, batch, namespace) for batch in batches]
        for done, future in enumerate(as_completed(futures), start=1):
            records.append(future.result())
            if done % 25 == 0 or done == len(futures):
                print(f"  {schema['name']} / {namespace}: {done}/{len(futures)} dávek", flush=True)
    names = [feature["name"] for feature in schema["features"]]
    features = pd.DataFrame([row for record in records for row in record["rows"]])
    features = features.set_index("id").reindex(columns=names).reindex(frame["id"])
    for feature in schema["features"]:
        dtype = {"integer": "Int64", "boolean": "boolean", "category": "string"}[feature["type"]]
        features[feature["name"]] = features[feature["name"]].astype(dtype)
    failed = [record for record in records if record.get("error")]
    stats = {
        "schema": schema["name"], "namespace": namespace, "rows": len(frame), "batches": len(batches),
        "failed_batches": len(failed), "batches_from_cache": sum(record["from_cache"] for record in records),
        "rows_without_answer": int(features.isna().all(axis=1).sum()),
        "requests": sum(record["requests"] for record in records),
        "api_seconds": round(sum(record["seconds"] for record in records), 1),
        "prompt_tokens": sum(record["prompt_tokens"] for record in records),
        "completion_tokens": sum(record["completion_tokens"] for record in records),
        "wall_seconds_this_run": round(time.perf_counter() - started, 1),
    }
    if failed:
        print(f"  Pozor: {len(failed)} dávek selhalo (např. {failed[0]['error']}). Spusťte buňku znovu.")
    return features, stats

In [10]:
SCHEMAS = {"manual": MANUAL_SCHEMA, "discovered": DISCOVERED_SCHEMA}
target = pilot_sample if RUN_MODE == "pilot" else texts
namespace = "pilot" if RUN_MODE == "pilot" else "full"
print(f"Zpracovávám {len(target):,} projektů ({RUN_MODE}).")

results, run_stats = {}, []
for schema_name in SCHEMAS_TO_RUN:
    schema = SCHEMAS[schema_name]
    features, stats = run_schema(schema, target[["id", "name", "blurb"]], namespace)
    results[schema_name] = features
    run_stats.append(stats)
    if stats["failed_batches"]:
        print(f"  {schema_name}: výstupní CSV neukládám, protože některé dávky selhaly. "
              "Spusťte tuto buňku znovu; hotové dávky se vezmou z cache.")
        continue

    output = features.add_prefix("llm_").reset_index()
    if RUN_MODE == "pilot":
        output = target[["id", "main_category", "name", "blurb"]].merge(output, on="id")
        output.to_csv(AI_DIR / f"pilot_features_{schema_name}.csv", index=False)
    else:
        output.to_csv(AI_DIR / f"llm_features_{schema_name}.csv", index=False)
    (AI_DIR / f"schema_used_{schema_name}.json").write_text(
        json.dumps(schema, ensure_ascii=False, indent=2), encoding="utf-8")

run_stats = pd.DataFrame(run_stats).set_index("schema")
run_stats.to_csv(AI_DIR / f"run_stats_{RUN_MODE}.csv")
display(run_stats)

Zpracovávám 26,635 projektů (full).


  manual / full: 25/1332 dávek


  manual / full: 50/1332 dávek


  manual / full: 75/1332 dávek


  manual / full: 100/1332 dávek


  manual / full: 125/1332 dávek


... (100 outputs left out)

  discovered / full: 1300/1332 dávek


  discovered / full: 1325/1332 dávek


  discovered / full: 1332/1332 dávek


,namespace,rows,batches,failed_batches,batches_from_cache,rows_without_answer,requests,api_seconds,prompt_tokens,completion_tokens,wall_seconds_this_run
schema,,,,,,,,,,,
manual,full,26635,1332,0,0,0,1332,14471.9,2051097,2085976,3619.3
discovered,full,26635,1332,0,0,0,1332,17926.2,2669145,2383474,4483.9


<a id="s5"></a>
## 5. Kontrola výsledků

<a id="s5-1"></a>
### 5.1 Chybějící hodnoty a rozložení

U každého příznaku sledujeme:
- jak často model nevrátil povolenou hodnotu,
- zda příznak vůbec rozlišuje. Pokud má jedna hodnota více než 95 % kampaní, je přínos příznaku pro model minimální.

In [11]:
quality_rows, distribution_rows = [], []
for schema_name, features in results.items():
    for column in features.columns:
        values = features[column]
        shares = values.value_counts(normalize=True)
        top_share = shares.iloc[0] if len(shares) else float("nan")
        quality_rows.append({
            "Schéma": schema_name, "Příznak": column,
            "Chybí": f"{values.isna().mean():.1%}", "Různých hodnot": values.nunique(),
            "Nejčastější": str(shares.index[0]) if len(shares) else None, "Její podíl": f"{top_share:.1%}",
            "Upozornění": "téměř konstantní" if top_share > 0.95 else "",
        })
        for value, share in shares.items():
            distribution_rows.append({"Schéma": schema_name, "Příznak": column, "Hodnota": str(value), "Podíl": f"{share:.1%}"})

display(pd.DataFrame(quality_rows).set_index(["Schéma", "Příznak"]))
if distribution_rows:
    with pd.option_context("display.max_rows", 200):
        display(pd.DataFrame(distribution_rows).set_index(["Schéma", "Příznak", "Hodnota"]))
else:
    print("Žádné platné odpovědi. Zkontrolujte chyby v části 4.")

Chybí  Různých hodnot  \
Schéma     Příznak                                             
manual     funding_purpose              0.0%               6   
           has_concrete_deliverable     0.0%               2   
           project_stage                0.0%               4   
           appeal_type                  0.0%               3   
           specificity                  0.0%               5   
           language                     0.2%               9   
discovered has_clear_value_proposition  0.0%               2   
           has_tangible_reward          0.0%               2   
           is_series_continuation       0.0%               2   
           creator_credibility_signal   0.1%               3   
           social_or_political_appeal   0.0%               2   
           technical_complexity         0.0%               5   
           has_emotional_hook           0.0%               2   

                                             Nejčastější Její podíl  \
Schéma     Příznak                                                    
manual     funding_purpose                 creative_work      58.3%   
           has_concrete_deliverable                 True      91.9%   
           project_stage                     in_progress      73.9%   
           appeal_type                  offer_to_backers      88.4%   
           specificity                                 3      42.9%   
           language                                   en      96.9%   
discovered has_clear_value_proposition              True      66.3%   
           has_tangible_reward                      True      84.3%   
           is_series_continuation                  False      89.0%   
           creator_credibility_signal               none      90.6%   
           social_or_political_appeal              False      90.7%   
           technical_complexity                        1      58.7%   
           has_emotional_hook                       True      51.8%   

                                              Upozornění  
Schéma     Příznak                                        
manual     funding_purpose                                
           has_concrete_deliverable                       
           project_stage                                  
           appeal_type                                    
           specificity                                    
           language                     téměř konstantní  
discovered has_clear_value_proposition                    
           has_tangible_reward                            
           is_series_continuation                         
           creator_credibility_signal                     
           social_or_political_appeal                     
           technical_complexity                           
           has_emotional_hook

Podíl
Schéma     Příznak                     Hodnota                         
manual     funding_purpose             creative_work              58.3%
                                       product                    32.0%
                                       business_operations         6.2%
                                       event                       1.4%
                                       charity_or_cause            1.1%
                                       other                       1.0%
           has_concrete_deliverable    True                       91.9%
                                       False                       8.1%
           project_stage               in_progress                73.9%
                                       idea                       12.4%
                                       finished                   11.4%
                                       unclear                     2.4%
           appeal_type                 offer_to_backers           88.4%
                                       mixed                       5.9%
                                       personal_plea               5.7%
           specificity                 3                          42.9%
                                       4                          34.4%
                                       2                          14.8%
                                       5                           4.7%
                                       1                           3.2%
           language                    en                         96.9%
                                       es                          1.1%
                                       fr                          0.6%
                                       de                          0.6%
                                       it                          0.4%
                                       other                       0.1%
                                       ja                          0.1%
                                       zh                          0.0%
                                       nl                          0.0%
discovered has_clear_value_proposition True                       66.3%
                                       False                      33.7%
           has_tangible_reward         True                       84.3%
                                       False                      15.7%
           is_series_continuation      False                      89.0%
                                       True                       11.0%
           creator_credibility_signal  none                       90.6%
                                       established_brand_or_team   8.3%
                                       awards_or_bestseller        1.1%
           social_or_political_appeal  False                      90.7%
                                       True                        9.3%
           technical_complexity        1                          58.7%
                                       2                          21.9%
                                       3                          12.4%
                                       4                           6.0%
                                       5                           0.9%
           has_emotional_hook          True                       51.8%
                                       False                      48.2%

<a id="s5-2"></a>
### 5.2 Ukázky

Několik kampaní s vyplněnými příznaky pro vizuální kontrolu, zda příznaky odpovídají textu.

In [12]:
examples = target[["id", "main_category", "name", "blurb"]].head(8).set_index("id")
for schema_name, features in results.items():
    examples = examples.join(features.add_prefix(f"{schema_name}:"))
with pd.option_context("display.max_colwidth", 120):
    display(examples.reset_index(drop=True).T)

,0,1,2,3,4,5,6,7
main_category,Comics,Comics,Comics,Fashion,Crafts,Theater,Music,Publishing
name,Dragon Grit #1-4: A Fantasy Western With Dragons!,Cütie Catcherz: One Shot,Horror Trading Card Tribute Issue of Gore Noir Magazine,ZODIAC EMOTIONS: Turning Astrology Into a Fashion Movement,Christmas Pen Kit: Your Custom DIY Woodturning Holiday Gift,New Life,Nuevo EP de corridos: La ciudad es nuestra,Remedy
blurb,A grizzled ex-outlaw takes a risky escort mission across the desert. Now with more ghosts.,"An immersive 60-page comic adventure that blends animation, music, pop-culture, and health education—like an animate...","Garbage Pail Kids, Magic The Gathering, Mars Attacks, Fright Flicks and MORE! We pay Tribute to all the cool Horror ...","Merging Astrology, Art, and Emotion into a Global Fashion Movement.","Capture the festive spirit! This DIY woodturning pen kit lets makers craft a unique, handmade Christmas gift or pers...",A theatre R&D project focussing on perinatal mental health.,Corridos sobre Monterrey.,Know Thyself.
manual:funding_purpose,creative_work,creative_work,creative_work,business_operations,product,creative_work,creative_work,other
manual:has_concrete_deliverable,True,True,True,False,True,False,True,False
manual:project_stage,in_progress,in_progress,in_progress,idea,in_progress,in_progress,in_progress,idea
manual:appeal_type,offer_to_backers,offer_to_backers,offer_to_backers,mixed,offer_to_backers,personal_plea,offer_to_backers,personal_plea
manual:specificity,3,4,3,2,4,2,2,1
manual:language,en,en,en,en,en,en,es,en
discovered:has_clear_value_proposition,False,True,False,False,True,True,False,False


<a id="s5-3"></a>
### 5.3 Konzistence (pouze v pilotu)

Část pilotu odešleme znovu v jiném pořadí, takže každá kampaň skončí v jiné dávce s jinými kampaněmi. Pokud se odpověď mění podle toho, s jakými kampaněmi je v dávce, je příznak nestabilní. U škál uvádíme i shodu s tolerancí ±1.

V plném běhu se tato kontrola neprovádí. V pilotu se binární příznaky a kategorie shodovaly v 85–100 % případů, škály přesně v 62–80 % a s tolerancí ±1 v 97–100 %.

In [13]:
if RUN_MODE == "pilot":
    rerun_frame = target.head(CONSISTENCY_SIZE).sample(frac=1, random_state=SEED + 7)
    consistency_rows = []
    for schema_name in SCHEMAS_TO_RUN:
        schema = SCHEMAS[schema_name]
        rerun, stats = run_schema(schema, rerun_frame[["id", "name", "blurb"]], "pilot_rerun")
        run_stats.loc[f"{schema_name} (opakování)"] = pd.Series(stats)
        first = results[schema_name].loc[rerun.index]
        for feature in schema["features"]:
            name = feature["name"]
            both = first[name].notna() & rerun[name].notna()
            same = first.loc[both, name].astype(str).eq(rerun.loc[both, name].astype(str)).mean()
            row = {"Schéma": schema_name, "Příznak": name, "Porovnáno": int(both.sum()), "Shoda": f"{same:.1%}"}
            if feature["type"] == "integer":
                close = (first.loc[both, name].astype(float) - rerun.loc[both, name].astype(float)).abs().le(1).mean()
                row["Shoda ±1"] = f"{close:.1%}"
            consistency_rows.append(row)
    display(pd.DataFrame(consistency_rows).fillna("").set_index(["Schéma", "Příznak"]))
else:
    print("Kontrola konzistence se dělá jen v pilotu.")

Kontrola konzistence se dělá jen v pilotu.


<a id="s5-4"></a>
### 5.4 Náklady

Endpoint je pro studenty bezplatný, náklady proto uvádíme v počtu dotazů, tokenech a čase, přepočtené i na 1 000 kampaní. V pilotu z těchto údajů odhadujeme dobu plného běhu se stejnou velikostí dávky a `MAX_WORKERS` souběžnými dotazy. Skutečná doba závisí na aktuálním vytížení sdíleného endpointu.

Plný běh trval 1,0 h (manual) a 1,25 h (discovered) se čtyřmi souběžnými dotazy, celkem 2 664 dotazů a přibližně 9,2 milionu tokenů.

In [14]:
cost_rows = []
full_rows = len(texts)
for schema_name, stats in run_stats.iterrows():
    if not stats["rows"]:
        continue
    per_row_seconds = stats["api_seconds"] / stats["rows"]
    cost_rows.append({
        "Schéma": schema_name,
        "Dotazů / 1 000 projektů": round(stats["requests"] / stats["rows"] * 1000, 1),
        "Tokenů / 1 000 projektů": round((stats["prompt_tokens"] + stats["completion_tokens"]) / stats["rows"] * 1000),
        "Čas API / 1 000 projektů (min)": round(per_row_seconds * 1000 / 60, 1),
        f"Odhad plného běhu: dotazů ({full_rows:,} projektů)": math.ceil(stats["requests"] / stats["rows"] * full_rows),
        "Odhad plného běhu (hod)": round(per_row_seconds * full_rows / MAX_WORKERS / 3600, 2),
    })
display(pd.DataFrame(cost_rows).set_index("Schéma"))
run_stats.to_csv(AI_DIR / f"run_stats_{RUN_MODE}.csv")

,Dotazů / 1 000 projektů,Tokenů / 1 000 projektů,Čas API / 1 000 projektů (min),"Odhad plného běhu: dotazů (26,635 projektů)",Odhad plného běhu (hod)
Schéma,,,,,
manual,50.0,155325,9.1,1332,1.00
discovered,50.0,189698,11.2,1332,1.24


<a id="s6"></a>
## 6. Postup a opakované spuštění

Po pilotu jsme ověřili:
1. tabulku [5.1](#s5-1): žádný příznak nemá vysoký podíl chybějících hodnot ani není téměř konstantní. Výjimkou je `language` (97 % angličtina), který jsme v dotazníku ponechali,
2. ukázky [5.2](#s5-2): hodnoty odpovídají textům,
3. konzistenci [5.3](#s5-3): u kategorií a binárních příznaků 85–100 %,
4. odhad v [5.4](#s5-4): očekávanou dobu plného běhu.

Na základě těchto kontrol jsme upravili dotazník navržený modelem ([3.2](#s3-2)) a oba dotazníky zafixovali. Prompt jsme po pilotu neměnili, `PROMPT_VERSION` zůstal `v1`.

Plný běh se spouští s nastavením `RUN_MODE = "full"`. Vzniknou soubory `_data/ai/llm_features_manual.csv` a `_data/ai/llm_features_discovered.csv` (klíč `id` a sloupce `llm_*`), které hlavní notebook připojuje podle `id`. Součástí odevzdání jsou tyto CSV soubory, použitá schémata (`schema_used_*.json`) a statistiky běhu (`run_stats_*.csv`). Pro běh hlavního notebooku není cache potřeba. Při opakovaném spuštění tohoto notebooku se stejnou cache se neodešlou žádné nové dotazy, API klíč je však potřeba i tak, a to kvůli kontrole spojení v [části 1](#s1).